# Heerlen-Aachen Annotated Steel Microstructure Dataset
## SegNet-Based Contour Detection + Three-Way Model Comparison

This notebook implements **SegNet** for MA island segmentation and compares
all three approaches head-to-head:

| # | Method | Type | Notes |
|---|---|---|---|
| 1 | Classical CV (original notebook) | Rule-based | Blur → Threshold → Erode/Dilate → `findContours` |
| 2 | U-Net | Deep learning | Encoder–decoder with skip connections |
| 3 | SegNet (this notebook) | Deep learning | Encoder–decoder with max-pool indices |

### SegNet vs U-Net — key architectural difference

Both are encoder–decoder networks, but they upsample differently:

```
U-Net    : Decoder receives FULL feature maps via skip connections
           (concatenation → keeps fine spatial detail, uses more memory)

SegNet   : Decoder receives only MAX-POOLING INDICES from the encoder
           (no feature map transfer → more memory-efficient, slightly lower
            boundary precision but faster inference)
```

SegNet's encoder is based on the **VGG-16** convolutional block structure.
Instead of storing full skip feature maps, each max-pool operation saves
only the 2-bit index of which pixel was the maximum.  The decoder uses
these indices for sparse upsampling before applying convolutions.

Contact: Based on dataset by Deniz Iren, PhD (deniz.iren@ou.nl)

---
## 0. File Paths & Settings

In [ ]:
## IMAGE FILES
path_folder_images_png  = 'PNG/'
path_folder_images_tiff = 'TIFF/'

## INPUTS FROM PREVIOUS NOTEBOOKS
path_file_annotations_morphology    = 'nature_scidata_heerlen_aachen_steel_morph.pickle'

## CLASSICAL CV RESULTS (from original auto_contour_detection notebook)
path_file_ClassicalEvaluations      = 'nature_scidata_dfEvaluation.pickle'

## U-NET RESULTS (from unet notebook)
path_file_UNetEvaluations           = 'nature_scidata_dfUNetEvaluation.pickle'

## SEGNET OUTPUTS (this notebook writes these)
path_file_SegNetContours            = 'nature_scidata_dfPOIPolygonSegNetContourShapely.pickle'
path_file_SegNetEvaluations         = 'nature_scidata_dfSegNetEvaluation.pickle'
path_segnet_weights                 = 'segnet_ma_island_weights.pth'

## TRAINING SETTINGS
IMAGE_HEIGHT      = 768
IMAGE_WIDTH       = 1024
PATCH_SIZE        = 256   # must be divisible by 2^5=32 for SegNet's 5 pool levels
TRAIN_SPLIT       = 0.80
BATCH_SIZE        = 4
NUM_EPOCHS        = 30
LEARNING_RATE     = 1e-4
DEVICE            = 'cuda'   # change to 'cpu' if no GPU

---
## 1. Imports

In [2]:
import os, math, pickle, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import Polygon as MplPolygon

import cv2
from shapely.geometry import Polygon, Point, MultiPolygon
import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
DEVICE = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')
print('Device  :', DEVICE)

PyTorch : 2.10.0+cu128
CUDA    : True
Device  : cuda


---
## 2. Load Annotation Data

In [ ]:
with open(path_file_annotations_morphology, 'rb') as fh:
    dfMorph = pickle.load(fh)

print(f'Total annotated MA islands : {len(dfMorph)}')
print(f'Unique images              : {dfMorph["image_url"].nunique()}')
dfMorph[['polygon_area', 'polygon_perimeter',
          'aspect_ratio', 'polygon_compactness']].describe()

FileNotFoundError: [Errno 2] No such file or directory: 'nature_scidata_heerlen_aachen_steel_morph.pickle'

---
## 3. Binary Mask Builder & Dataset

Identical helper to the U-Net notebook so both models train on
exactly the same data splits and masks.

In [ ]:
def build_mask_for_image(image_url, df, h=IMAGE_HEIGHT, w=IMAGE_WIDTH):
    """Paint all expert polygons for one image onto a binary mask."""
    mask = np.zeros((h, w), dtype=np.uint8)
    for _, row in df[df['image_url'] == image_url].iterrows():
        coords = np.array(row['polygon'], dtype=np.int32)
        cv2.fillPoly(mask, [coords.reshape(-1, 1, 2)], color=255)
    return mask


class MAIslandDataset(Dataset):
    """
    Random-patch dataset — identical API to the U-Net notebook
    so train/val splits can be shared for a fair comparison.
    """
    def __init__(self, image_urls, df, img_dir,
                 patch_size=PATCH_SIZE, patches_per_image=8, augment=True):
        self.image_urls        = image_urls
        self.df                = df
        self.img_dir           = img_dir
        self.patch_size        = patch_size
        self.patches_per_image = patches_per_image
        self.augment           = augment
        self.images, self.masks = {}, {}

        print('Loading images and building masks …')
        for url in tqdm.tqdm(self.image_urls):
            img = cv2.imread(self.img_dir + url, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                self.images[url] = img
                self.masks[url]  = build_mask_for_image(url, self.df)
        self.valid_urls = list(self.images.keys())

    def __len__(self):
        return len(self.valid_urls) * self.patches_per_image

    def __getitem__(self, idx):
        url  = self.valid_urls[idx % len(self.valid_urls)]
        img  = self.images[url].copy().astype(np.float32) / 255.0
        mask = self.masks[url].copy().astype(np.float32)  / 255.0
        h, w = img.shape
        p    = self.patch_size
        top  = random.randint(0, h - p)
        left = random.randint(0, w - p)
        img_p  = img [top:top+p, left:left+p]
        mask_p = mask[top:top+p, left:left+p]
        if self.augment:
            if random.random() > 0.5:
                img_p  = np.fliplr(img_p).copy();  mask_p = np.fliplr(mask_p).copy()
            if random.random() > 0.5:
                img_p  = np.flipud(img_p).copy();  mask_p = np.flipud(mask_p).copy()
            k = random.randint(0, 3)
            img_p  = np.rot90(img_p,  k).copy()
            mask_p = np.rot90(mask_p, k).copy()
        return (torch.from_numpy(img_p ).unsqueeze(0),
                torch.from_numpy(mask_p).unsqueeze(0))


# ── Same seed & split as U-Net notebook ──────────────────────────────────────
all_urls = dfMorph['image_url'].unique().tolist()
random.seed(42)
random.shuffle(all_urls)
n_train    = int(len(all_urls) * TRAIN_SPLIT)
train_urls = all_urls[:n_train]
val_urls   = all_urls[n_train:]
print(f'Train images: {len(train_urls)} | Val images: {len(val_urls)}')

train_ds = MAIslandDataset(train_urls, dfMorph, path_folder_images_png,
                            patches_per_image=10, augment=True)
val_ds   = MAIslandDataset(val_urls,   dfMorph, path_folder_images_png,
                            patches_per_image=4,  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Train images: 1364 | Val images: 341
Loading images and building masks …


100%|██████████| 1364/1364 [00:35<00:00, 38.40it/s]


Loading images and building masks …


100%|██████████| 341/341 [00:08<00:00, 39.23it/s]

Train batches: 3410 | Val batches: 341


---
## 4. SegNet Architecture

SegNet *(Badrinarayanan et al., 2017)* uses a VGG-16-style encoder and a
mirrored decoder.  The crucial difference from U-Net is **max-pool index
upsampling**: instead of transferring full feature maps (skip connections),
the encoder stores only the 2-bit position of each pooling window's maximum.
The decoder uses these indices to place activated pixels back into a sparse
feature map before convolving.

```
Input  (1 × 256 × 256)
  │
  ├─ Enc Block 1: Conv64 → Conv64  → MaxPool → save idx1
  ├─ Enc Block 2: Conv128→ Conv128 → MaxPool → save idx2
  ├─ Enc Block 3: Conv256→ Conv256 → MaxPool → save idx3
  ├─ Enc Block 4: Conv512→ Conv512 → MaxPool → save idx4
  ├─ Enc Block 5: Conv512→ Conv512 → MaxPool → save idx5
  │
  ├─ Dec Block 5: Unpool(idx5) → Conv512→ Conv512
  ├─ Dec Block 4: Unpool(idx4) → Conv512→ Conv256
  ├─ Dec Block 3: Unpool(idx3) → Conv256→ Conv128
  ├─ Dec Block 2: Unpool(idx2) → Conv128→  Conv64
  ├─ Dec Block 1: Unpool(idx1) → Conv64 →  Conv64
  │
Output (1 × 256 × 256)  sigmoid
```

In [ ]:
class ConvBNReLU(nn.Module):
    """Conv2d → BatchNorm2d → ReLU (standard SegNet building block)."""
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class SegNet(nn.Module):
    """
    SegNet for binary segmentation of grayscale images.

    Key detail: nn.MaxPool2d(return_indices=True) returns the flat index
    of the maximum value in each pooling window.  nn.MaxUnpool2d uses
    these indices to place values back at their original spatial position,
    zeroing all other locations.  This is the defining SegNet mechanism.
    """
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        # ── Pooling / Unpooling ──────────────────────────────────────────
        self.pool   = nn.MaxPool2d(2, 2, return_indices=True)
        self.unpool = nn.MaxUnpool2d(2, 2)

        # ── Encoder blocks (VGG-16 inspired) ────────────────────────────
        self.enc1 = nn.Sequential(ConvBNReLU(in_channels, 64),
                                   ConvBNReLU(64, 64))
        self.enc2 = nn.Sequential(ConvBNReLU(64,  128),
                                   ConvBNReLU(128, 128))
        self.enc3 = nn.Sequential(ConvBNReLU(128, 256),
                                   ConvBNReLU(256, 256))
        self.enc4 = nn.Sequential(ConvBNReLU(256, 512),
                                   ConvBNReLU(512, 512))
        self.enc5 = nn.Sequential(ConvBNReLU(512, 512),
                                   ConvBNReLU(512, 512))

        # ── Decoder blocks (mirror of encoder, NO skip connections) ──────
        self.dec5 = nn.Sequential(ConvBNReLU(512, 512),
                                   ConvBNReLU(512, 512))
        self.dec4 = nn.Sequential(ConvBNReLU(512, 512),
                                   ConvBNReLU(512, 256))
        self.dec3 = nn.Sequential(ConvBNReLU(256, 256),
                                   ConvBNReLU(256, 128))
        self.dec2 = nn.Sequential(ConvBNReLU(128, 128),
                                   ConvBNReLU(128, 64))
        self.dec1 = nn.Sequential(ConvBNReLU(64,  64),
                                   ConvBNReLU(64,  64))

        # ── Final classification layer ───────────────────────────────────
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # ── Encoder (save pool indices at each level) ────────────────────
        x1 = self.enc1(x);  x, idx1 = self.pool(x1)   # 256→128
        x2 = self.enc2(x);  x, idx2 = self.pool(x2)   # 128→ 64
        x3 = self.enc3(x);  x, idx3 = self.pool(x3)   #  64→ 32
        x4 = self.enc4(x);  x, idx4 = self.pool(x4)   #  32→ 16
        x5 = self.enc5(x);  x, idx5 = self.pool(x5)   #  16→  8

        # ── Decoder (sparse upsample using saved indices) ────────────────
        x = self.unpool(x,  idx5, output_size=x5.size()); x = self.dec5(x)
        x = self.unpool(x,  idx4, output_size=x4.size()); x = self.dec4(x)
        x = self.unpool(x,  idx3, output_size=x3.size()); x = self.dec3(x)
        x = self.unpool(x,  idx2, output_size=x2.size()); x = self.dec2(x)
        x = self.unpool(x,  idx1, output_size=x1.size()); x = self.dec1(x)

        return torch.sigmoid(self.final(x))


segnet = SegNet(in_channels=1, out_channels=1).to(DEVICE)
n_params = sum(p.numel() for p in segnet.parameters() if p.requires_grad)
print(f'SegNet parameters: {n_params:,}')
print(segnet)

SegNet parameters: 18,849,025
SegNet(
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (unpool): MaxUnpool2d(kernel_size=(2, 2), stride=(2, 2), padding=(0, 0))
  (enc1): Sequential(
    (0): ConvBNReLU(
      (block): Sequential(
        (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
    )
    (1): ConvBNReLU(
      (block): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
    )
  )
  (enc2): Sequential(
    (0): ConvBNReLU(
      (block): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affi

---
## 5. Loss Function (identical to U-Net notebook for fair comparison)

In [ ]:
class DiceBCELoss(nn.Module):
    """Dice + Binary Cross-Entropy combined loss."""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce    = nn.BCELoss()

    def forward(self, pred, target):
        bce_loss    = self.bce(pred, target)
        p, t        = pred.view(-1), target.view(-1)
        intersection= (p * t).sum()
        dice_loss   = 1 - (2 * intersection + self.smooth) / \
                          (p.sum() + t.sum() + self.smooth)
        return bce_loss + dice_loss


criterion = DiceBCELoss()
optimizer = optim.Adam(segnet.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5
)

---
## 6. Training Loop

In [ ]:
def dice_score(pred_bin, target):
    smooth = 1.0
    p = pred_bin.view(-1).float()
    t = target.view(-1).float()
    return ((2 * (p * t).sum() + smooth) / (p.sum() + t.sum() + smooth)).item()


train_losses, val_losses, val_dices = [], [], []
epoch_times = []
best_val_dice = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ── Train ────────────────────────────────────────────────────────────
    segnet.train()
    run_loss = 0.0
    for imgs, masks in tqdm.tqdm(train_loader, desc=f'Ep {epoch:02d} Train'):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = segnet(imgs)
        loss  = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        run_loss += loss.item()
    train_losses.append(run_loss / len(train_loader))

    # ── Validate ─────────────────────────────────────────────────────────
    segnet.eval()
    run_vloss = run_dice = 0.0
    with torch.no_grad():
        for imgs, masks in tqdm.tqdm(val_loader, desc=f'Ep {epoch:02d} Val  '):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = segnet(imgs)
            run_vloss += criterion(preds, masks).item()
            run_dice  += dice_score((preds > 0.5).float(), masks)
    val_losses.append(run_vloss / len(val_loader))
    val_dices.append(run_dice  / len(val_loader))
    scheduler.step(val_losses[-1])

    elapsed = time.time() - t0
    epoch_times.append(elapsed)
    print(f'Ep {epoch:02d}/{NUM_EPOCHS}  '
          f'TrLoss={train_losses[-1]:.4f}  '
          f'VaLoss={val_losses[-1]:.4f}  '
          f'VaDice={val_dices[-1]:.4f}  '
          f'Time={elapsed:.1f}s')

    if val_dices[-1] > best_val_dice:
        best_val_dice = val_dices[-1]
        torch.save(segnet.state_dict(), path_segnet_weights)
        print(f'  ✓ Best model saved (Val Dice = {best_val_dice:.4f})')

print('\nTraining complete. Best Val Dice:', round(best_val_dice, 4))
print(f'Avg epoch time: {sum(epoch_times)/len(epoch_times):.1f}s')

Ep 01 Train:   0%|          | 0/3410 [00:00<?, ?it/s]d:\Btp\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# Plot SegNet training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ep = range(1, len(train_losses) + 1)

ax1.plot(ep, train_losses, label='Train', color='steelblue')
ax1.plot(ep, val_losses,   label='Val',   color='darkorange')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('DiceBCE Loss')
ax1.set_title('SegNet — Training & Validation Loss')
ax1.legend()

ax2.plot(ep, val_dices, color='green', label='SegNet Val Dice')
ax2.axhline(0.35, color='red',    linestyle='--', label='Classical CV (IOU 0.35)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Dice Score')
ax2.set_title('SegNet Val Dice vs Classical Baseline')
ax2.legend()

plt.tight_layout(); plt.show()

---
## 7. Inference with POI Guidance

Identical sliding-window + `pointPolygonTest` logic used in both the
original notebook and the U-Net notebook, ensuring the evaluation
pipeline is identical across all three methods.

In [ ]:
segnet.load_state_dict(torch.load(path_segnet_weights, map_location=DEVICE))
segnet.eval()
print('Best SegNet weights loaded.')


def predict_full_image(img_gray, model, patch_size=PATCH_SIZE, overlap=32):
    """
    Sliding-window inference over a full SEM image.
    Overlapping tile predictions are averaged to reduce border artefacts.
    """
    h, w     = img_gray.shape
    prob_map = np.zeros((h, w), dtype=np.float32)
    count    = np.zeros((h, w), dtype=np.float32)
    step     = patch_size - overlap
    img_norm = img_gray.astype(np.float32) / 255.0

    with torch.no_grad():
        for top in range(0, h - patch_size + 1, step):
            for left in range(0, w - patch_size + 1, step):
                patch = img_norm[top:top+patch_size, left:left+patch_size]
                t = torch.from_numpy(patch).unsqueeze(0).unsqueeze(0).to(DEVICE)
                pred = model(t).squeeze().cpu().numpy()
                prob_map[top:top+patch_size, left:left+patch_size] += pred
                count   [top:top+patch_size, left:left+patch_size] += 1.0

    count[count == 0] = 1
    return prob_map / count


def find_segnet_contour_around_point(prob_map, point_tuple, threshold=0.5):
    """
    Binarise SegNet probability map, find connected components, return the
    Shapely polygon containing the expert POI.
    (Same pointPolygonTest logic as original notebook and U-Net notebook.)
    """
    binary = (prob_map > threshold).astype(np.uint8) * 255
    contours, _ = cv2.findContours(binary, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    for cx in contours:
        if cv2.pointPolygonTest(cx, tuple(point_tuple), False) > 0:
            pts = np.squeeze(cx)
            if pts.ndim == 2 and len(pts) >= 3:
                poly = Polygon(pts)
                return poly if poly.is_valid else poly.buffer(0)
    return None

In [ ]:
# Run SegNet inference over all annotated MA islands
prob_map_cache = {}
results = []

print('Running SegNet inference …')
for _, row in tqdm.tqdm(dfMorph.iterrows(), total=len(dfMorph)):
    url   = row['image_url']
    point = row['point']

    if url not in prob_map_cache:
        img = cv2.imread(path_folder_images_png + url, cv2.IMREAD_GRAYSCALE)
        prob_map_cache[url] = predict_full_image(img, segnet) if img is not None else None

    pm      = prob_map_cache[url]
    contour = find_segnet_contour_around_point(pm, tuple(point)) if pm is not None else None

    results.append([url, row['point'], row['polygon'],
                    row['point_shapely'], row['poly_shapely'], contour])

dfSegNetContours = pd.DataFrame(
    results,
    columns=['image_url', 'point', 'polygon',
             'point_shapely', 'poly_shapely', 'contour_polygon_shapely']
)
n_det = dfSegNetContours['contour_polygon_shapely'].notna().sum()
print(f'Detected : {n_det} / {len(dfSegNetContours)}')
print(f'Missed   : {len(dfSegNetContours) - n_det}')

In [ ]:
## Save SegNet contours (uncomment to write):
# with open(path_file_SegNetContours, 'wb') as fh:
#     pickle.dump(dfSegNetContours, fh, protocol=pickle.HIGHEST_PROTOCOL)
print('Done.')

---
## 8. Evaluation — IOU

Same `polygonCompareAgainstGold` + `calculateIOU` functions as the
original and U-Net notebooks.

In [ ]:
def polygonCompareAgainstGold(contourPoly, goldPoly):
    inters_area = uni_area = iou = 0.0
    if contourPoly.is_valid and goldPoly.is_valid:
        inters = contourPoly.intersection(goldPoly)
        uni    = contourPoly.union(goldPoly)
        inters_area, uni_area = inters.area, uni.area
        if uni_area > 0:
            iou = inters_area / uni_area
    return goldPoly.area, contourPoly.area, inters_area, uni_area, iou


def calculateIOU(dfShapely):
    retList = []
    for _, row in dfShapely.iterrows():
        goldPoly = row['poly_shapely']
        contour  = row['contour_polygon_shapely']
        goldArea = contourArea = iou = 0.0
        if contour is not None and goldPoly.is_valid and contour.is_valid:
            goldArea, contourArea, _, _, iou = \
                polygonCompareAgainstGold(contour, goldPoly)
        elif goldPoly.is_valid:
            goldArea = goldPoly.area
        retList.append([
            row['image_url'], row['point'], row['polygon'],
            row['point_shapely'], row['poly_shapely'],
            row['contour_polygon_shapely'],
            goldArea, contourArea, iou
        ])
    return pd.DataFrame(retList, columns=[
        'image_url', 'point', 'polygon', 'point_shapely', 'poly_shapely',
        'contour_polygon_shapely', 'area_poly', 'area_contour', 'IOU'
    ])


dfSegNetEval = calculateIOU(dfSegNetContours)

n_det  = (dfSegNetEval['IOU'] > 0).sum()
print(f'Mean IOU   : {dfSegNetEval["IOU"].mean():.4f}')
print(f'Median IOU : {dfSegNetEval["IOU"].median():.4f}')
print(f'Detection  : {n_det}/{len(dfSegNetEval)} ({n_det/len(dfSegNetEval)*100:.1f}%)')

In [ ]:
## Save evaluation (uncomment to write):
# with open(path_file_SegNetEvaluations, 'wb') as fh:
#     pickle.dump(dfSegNetEval, fh, protocol=pickle.HIGHEST_PROTOCOL)
print('Done.')

---
## 9. Three-Way Model Comparison

This section loads evaluation results from all three methods and
produces a comprehensive side-by-side comparison.

In [ ]:
# ── Load all three evaluation dataframes ─────────────────────────────────────

# 1. Classical CV (from original notebook)
with open(path_file_ClassicalEvaluations, 'rb') as fh:
    dfClassical = pickle.load(fh)

# 2. U-Net (from U-Net notebook)
with open(path_file_UNetEvaluations, 'rb') as fh:
    dfUNet = pickle.load(fh)

# 3. SegNet (computed in this notebook)
dfSegNet = dfSegNetEval.copy()

# Helper: summary stats for one model
def summarise(df, name):
    iou = df['IOU']
    detected = (iou > 0).sum()
    return {
        'Model'         : name,
        'Mean IOU'      : round(iou.mean(), 4),
        'Median IOU'    : round(iou.median(), 4),
        'Std IOU'       : round(iou.std(), 4),
        'IOU > 0.5 (%)'  : round((iou > 0.5).sum() / len(iou) * 100, 1),
        'IOU > 0.75 (%)' : round((iou > 0.75).sum() / len(iou) * 100, 1),
        'Detected (%)'  : round(detected / len(iou) * 100, 1),
        'n'             : len(iou),
    }

summary = pd.DataFrame([
    summarise(dfClassical, 'Classical CV'),
    summarise(dfUNet,      'U-Net'),
    summarise(dfSegNet,    'SegNet'),
]).set_index('Model')

print('\n=== Three-Way Model Comparison ===')
print(summary.to_string())
summary

### 9.1 IOU Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

models = [
    (dfClassical, 'Classical CV',  '#d62728'),
    (dfUNet,      'U-Net',         '#1f77b4'),
    (dfSegNet,    'SegNet',        '#2ca02c'),
]

for ax, (df, name, color) in zip(axes, models):
    iou = df['IOU']
    ax.hist(iou, bins=40, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(iou.mean(),   color='black', linestyle='-',  linewidth=2,
               label=f'Mean = {iou.mean():.3f}')
    ax.axvline(iou.median(), color='black', linestyle='--', linewidth=1.5,
               label=f'Median = {iou.median():.3f}')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('IOU')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Number of MA islands')
plt.suptitle('IOU Distribution — All Three Methods', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 9.2 Overlapping Density Plot

In [ ]:
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(10, 5))
x_vals  = np.linspace(0, 1, 300)

for df, name, color in models:
    iou = df['IOU'].values
    kde = gaussian_kde(iou, bw_method=0.12)
    ax.plot(x_vals, kde(x_vals), color=color, linewidth=2.5, label=name)
    ax.fill_between(x_vals, kde(x_vals), alpha=0.15, color=color)
    ax.axvline(iou.mean(), color=color, linestyle=':', linewidth=1.5)

ax.set_xlabel('IOU', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('IOU Density — Classical CV vs U-Net vs SegNet', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

### 9.3 Bar Chart — Key Metrics Side-by-Side

In [ ]:
metrics = ['Mean IOU', 'Median IOU', 'IOU > 0.5 (%)', 'IOU > 0.75 (%)', 'Detected (%)']
model_names  = summary.index.tolist()
model_colors = ['#d62728', '#1f77b4', '#2ca02c']

fig, axes = plt.subplots(1, len(metrics), figsize=(20, 5))

for ax, metric in zip(axes, metrics):
    vals = summary[metric].values
    bars = ax.bar(model_names, vals, color=model_colors, edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(vals) * 1.25)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.02,
                f'{val:.3f}' if metric in ('Mean IOU', 'Median IOU', 'Std IOU') else f'{val:.1f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Model Comparison — Key Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 9.4 IOU Category Breakdown (Stacked Bar)

In [ ]:
def iou_category_pcts(df):
    iou = df['IOU']
    n   = len(iou)
    return {
        'No detection (0)'    : (iou == 0).sum()           / n * 100,
        'Poor  (0–0.25)'      : ((iou > 0)    & (iou <= 0.25)).sum() / n * 100,
        'Fair  (0.25–0.50)'   : ((iou > 0.25) & (iou <= 0.50)).sum() / n * 100,
        'Good  (0.50–0.75)'   : ((iou > 0.50) & (iou <= 0.75)).sum() / n * 100,
        'Great (0.75–1.00)'   : (iou > 0.75).sum()         / n * 100,
    }

cat_colors = ['#7f7f7f', '#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']
cat_data = {
    'Classical CV' : iou_category_pcts(dfClassical),
    'U-Net'        : iou_category_pcts(dfUNet),
    'SegNet'       : iou_category_pcts(dfSegNet),
}
cat_df = pd.DataFrame(cat_data).T

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(3)
for col, color in zip(cat_df.columns, cat_colors):
    vals = cat_df[col].values
    bars = ax.bar(cat_df.index, vals, bottom=bottom, color=color, label=col, edgecolor='white')
    for bar, val in zip(bars, vals):
        if val > 3:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_y() + bar.get_height() / 2,
                    f'{val:.1f}%', ha='center', va='center',
                    fontsize=8, color='white', fontweight='bold')
    bottom += vals

ax.set_ylabel('% of MA Islands', fontsize=12)
ax.set_title('IOU Category Breakdown — All Three Methods', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 115)
plt.tight_layout()
plt.show()

### 9.5 Per-Island Scatter — SegNet vs U-Net IOU

In [ ]:
# Align the three dataframes on image_url + point so we compare the same islands
dfClassical['key'] = dfClassical['image_url'] + '_' + dfClassical['point'].astype(str)
dfUNet['key']      = dfUNet['image_url']      + '_' + dfUNet['point'].astype(str)
dfSegNet['key']    = dfSegNet['image_url']    + '_' + dfSegNet['point'].astype(str)

merged = (dfUNet[['key', 'IOU']].rename(columns={'IOU': 'IOU_UNet'})
          .merge(dfSegNet[['key', 'IOU']].rename(columns={'IOU': 'IOU_SegNet'}), on='key')
          .merge(dfClassical[['key', 'IOU']].rename(columns={'IOU': 'IOU_Classical'}), on='key'))

print(f'Matched islands: {len(merged)}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pairs = [
    ('IOU_UNet', 'IOU_SegNet',    'U-Net IOU',     'SegNet IOU',    '#1f77b4'),
    ('IOU_Classical', 'IOU_UNet', 'Classical CV', 'U-Net IOU',     '#d62728'),
    ('IOU_Classical', 'IOU_SegNet','Classical CV', 'SegNet IOU',    '#2ca02c'),
]

for ax, (xcol, ycol, xlabel, ylabel, color) in zip(axes, pairs):
    ax.scatter(merged[xcol], merged[ycol], alpha=0.25, s=6, color=color)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Equal performance')
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    n_better = (merged[ycol] > merged[xcol]).sum()
    ax.set_title(f'{ylabel} vs {xlabel}\n'
                 f'({n_better}/{len(merged)} islands: {ylabel.split()[0]} wins)',
                 fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.suptitle('Per-Island IOU Scatter (points above diagonal = row model wins)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 9.6 Box Plot — IOU Distribution per Model

In [ ]:
import matplotlib.pyplot as plt

iou_data   = [dfClassical['IOU'].values, dfUNet['IOU'].values, dfSegNet['IOU'].values]
iou_labels = ['Classical CV', 'U-Net', 'SegNet']
iou_colors = ['#d62728',      '#1f77b4', '#2ca02c']

fig, ax = plt.subplots(figsize=(8, 6))
bp = ax.boxplot(iou_data, labels=iou_labels, patch_artist=True,
                medianprops=dict(color='white', linewidth=2),
                whiskerprops=dict(linewidth=1.5),
                capprops=dict(linewidth=1.5),
                flierprops=dict(marker='o', markersize=2, alpha=0.3))

for patch, color in zip(bp['boxes'], iou_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

# Add mean markers
for i, vals in enumerate(iou_data):
    ax.scatter(i + 1, np.mean(vals), marker='D', color='white',
               edgecolors=iou_colors[i], s=60, zorder=5, label='Mean' if i == 0 else '')

ax.set_ylabel('IOU Score', fontsize=12)
ax.set_title('IOU Distribution — Box Plot Comparison', fontsize=13, fontweight='bold')
ax.legend(['Mean (diamond)'], fontsize=9)
ax.set_ylim(0, 1.05)
ax.axhline(0, color='grey', linewidth=0.5, linestyle=':')
plt.tight_layout()
plt.show()

---
## 10. Visual Side-by-Side: Same Island, Three Methods

In [ ]:
def visualize_three_way(merged_df, df_classical, df_unet, df_segnet,
                        img_dir, n_samples=4, min_avg_iou=0.2):
    """
    For each sampled MA island, show the SEM image crop with three
    overlays: Expert polygon (red), U-Net (blue), SegNet (green).
    Classical CV contours are shown as orange if available.
    """
    # Filter to islands where at least two models detected something
    eligible = merged_df[
        ((merged_df['IOU_UNet'] + merged_df['IOU_SegNet']) / 2) >= min_avg_iou
    ]
    samples = eligible.sample(min(n_samples, len(eligible)), random_state=7)

    fig, axes = plt.subplots(n_samples, 4, figsize=(20, n_samples * 4))
    if n_samples == 1:
        axes = axes[np.newaxis, :]

    col_titles = ['SEM Image', 'Classical CV', 'U-Net', 'SegNet']
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=12, fontweight='bold')

    for row_idx, (_, sample) in enumerate(samples.iterrows()):
        key = sample['key']

        # Fetch rows from each model
        r_cl  = df_classical[df_classical['key'] == key].iloc[0]
        r_un  = df_unet     [df_unet['key']      == key].iloc[0]
        r_sn  = df_segnet   [df_segnet['key']    == key].iloc[0]

        img = cv2.imread(img_dir + r_un['image_url'], cv2.IMREAD_GRAYSCALE)

        # Crop region around expert polygon
        bounds = r_un['poly_shapely'].bounds
        pad = 30
        x1 = max(0, int(bounds[0]) - pad);  y1 = max(0, int(bounds[1]) - pad)
        x2 = min(img.shape[1], int(bounds[2]) + pad)
        y2 = min(img.shape[0], int(bounds[3]) + pad)
        crop = img[y1:y2, x1:x2]

        def shift(coords): return [(x - x1, y - y1) for x, y in coords]

        def add_poly(ax, shapely_poly, color, ls='-', lw=1.8):
            if shapely_poly is None: return
            geoms = [shapely_poly] if shapely_poly.geom_type == 'Polygon' \
                    else list(shapely_poly.geoms)
            for g in geoms:
                ax.add_patch(MplPolygon(shift(list(g.exterior.coords)),
                             closed=True, edgecolor=color,
                             facecolor='none', linewidth=lw, linestyle=ls))

        expert_poly = r_un['poly_shapely']

        # Col 0 — raw image
        axes[row_idx, 0].imshow(crop, cmap='gray')
        add_poly(axes[row_idx, 0], expert_poly, 'red')
        axes[row_idx, 0].set_ylabel(
            r_un['image_url'][:20] + '…', fontsize=7, rotation=0, labelpad=80)

        # Col 1 — Classical CV
        axes[row_idx, 1].imshow(crop, cmap='gray')
        add_poly(axes[row_idx, 1], expert_poly, 'red', ls='--', lw=1.2)
        cl_contour = r_cl.get('contour_polygon_shapely', None)
        add_poly(axes[row_idx, 1], cl_contour, 'orange')
        axes[row_idx, 1].set_title(f'IOU={r_cl["IOU"]:.3f}', fontsize=8)

        # Col 2 — U-Net
        axes[row_idx, 2].imshow(crop, cmap='gray')
        add_poly(axes[row_idx, 2], expert_poly, 'red', ls='--', lw=1.2)
        add_poly(axes[row_idx, 2], r_un['contour_polygon_shapely'], 'deepskyblue')
        axes[row_idx, 2].set_title(f'IOU={r_un["IOU"]:.3f}', fontsize=8)

        # Col 3 — SegNet
        axes[row_idx, 3].imshow(crop, cmap='gray')
        add_poly(axes[row_idx, 3], expert_poly, 'red', ls='--', lw=1.2)
        add_poly(axes[row_idx, 3], r_sn['contour_polygon_shapely'], 'limegreen')
        axes[row_idx, 3].set_title(f'IOU={r_sn["IOU"]:.3f}', fontsize=8)

        for ax in axes[row_idx]:
            ax.axis('off')

    # Shared legend
    legend_handles = [
        mpatches.Patch(edgecolor='red',        facecolor='none', label='Expert polygon (gold standard)'),
        mpatches.Patch(edgecolor='orange',     facecolor='none', label='Classical CV contour'),
        mpatches.Patch(edgecolor='deepskyblue',facecolor='none', label='U-Net contour'),
        mpatches.Patch(edgecolor='limegreen',  facecolor='none', label='SegNet contour'),
    ]
    fig.legend(handles=legend_handles, loc='lower center',
               ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.01))
    plt.suptitle('Three-Way Visual Comparison — Same MA Island',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


visualize_three_way(merged, dfClassical, dfUNet, dfSegNet,
                    path_folder_images_png, n_samples=4)

---
## 11. Architecture Comparison: SegNet vs U-Net

In [ ]:
from torchinfo import summary as model_summary   # pip install torchinfo

dummy = torch.zeros(1, 1, PATCH_SIZE, PATCH_SIZE).to(DEVICE)

print('\n=== SegNet ===')
model_summary(segnet, input_size=(1, 1, PATCH_SIZE, PATCH_SIZE), device=DEVICE)

# Load U-Net for comparison (re-instantiate architecture)
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=(64, 128, 256, 512)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups   = nn.ModuleList()
        self.pool  = nn.MaxPool2d(2, 2)
        ch = in_ch
        for f in features:
            self.downs.append(DoubleConv(ch, f)); ch = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f*2, f, 2, 2))
            self.ups.append(DoubleConv(f*2, f))
        self.final = nn.Conv2d(features[0], out_ch, 1)
    def forward(self, x):
        skips = []
        for d in self.downs:
            x = d(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x); skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x); s = skips[i//2]
            if x.shape != s.shape: x = TF.resize(x, s.shape[2:])
            x = torch.cat([s, x], 1); x = self.ups[i+1](x)
        return torch.sigmoid(self.final(x))

unet_ref = UNet().to(DEVICE)
print('\n=== U-Net ===')
model_summary(unet_ref, input_size=(1, 1, PATCH_SIZE, PATCH_SIZE), device=DEVICE)

In [ ]:
# ── Speed benchmark ───────────────────────────────────────────────────────────
import time
N_RUNS = 50
dummy  = torch.zeros(1, 1, PATCH_SIZE, PATCH_SIZE).to(DEVICE)

def benchmark(model, name, n=N_RUNS):
    model.eval()
    with torch.no_grad():
        # Warm-up
        for _ in range(5): model(dummy)
        t0 = time.perf_counter()
        for _ in range(n): model(dummy)
        elapsed = (time.perf_counter() - t0) / n * 1000
    print(f'{name}: {elapsed:.2f} ms / patch')
    return elapsed

t_unet   = benchmark(unet_ref, 'U-Net')
t_segnet = benchmark(segnet,   'SegNet')

print(f'\nSegNet is {t_unet/t_segnet:.2f}x faster than U-Net per patch'
      if t_segnet < t_unet else
      f'\nU-Net is  {t_segnet/t_unet:.2f}x faster than SegNet per patch')

---
## 12. Final Summary Table & Recommendation

In [ ]:
unet_params   = sum(p.numel() for p in unet_ref.parameters())
segnet_params = sum(p.numel() for p in segnet.parameters())

final_summary = pd.DataFrame({
    'Metric'            : ['Mean IOU', 'Median IOU', 'Detection rate (%)',
                           'IOU > 0.5 (%)', 'IOU > 0.75 (%)', 'Parameters (M)',
                           'Inference speed (ms/patch)', 'Skip connections',
                           'Upsampling method'],
    'Classical CV'      : [round(dfClassical['IOU'].mean(),4),
                           round(dfClassical['IOU'].median(),4),
                           round((dfClassical['IOU']>0).sum()/len(dfClassical)*100,1),
                           round((dfClassical['IOU']>0.5).sum()/len(dfClassical)*100,1),
                           round((dfClassical['IOU']>0.75).sum()/len(dfClassical)*100,1),
                           'N/A', 'N/A', 'N/A', 'Threshold + Morphology'],
    'U-Net'             : [round(dfUNet['IOU'].mean(),4),
                           round(dfUNet['IOU'].median(),4),
                           round((dfUNet['IOU']>0).sum()/len(dfUNet)*100,1),
                           round((dfUNet['IOU']>0.5).sum()/len(dfUNet)*100,1),
                           round((dfUNet['IOU']>0.75).sum()/len(dfUNet)*100,1),
                           round(unet_params/1e6, 2), round(t_unet,2),
                           'Yes (full feature maps)', 'Transposed convolution'],
    'SegNet'            : [round(dfSegNet['IOU'].mean(),4),
                           round(dfSegNet['IOU'].median(),4),
                           round((dfSegNet['IOU']>0).sum()/len(dfSegNet)*100,1),
                           round((dfSegNet['IOU']>0.5).sum()/len(dfSegNet)*100,1),
                           round((dfSegNet['IOU']>0.75).sum()/len(dfSegNet)*100,1),
                           round(segnet_params/1e6, 2), round(t_segnet,2),
                           'No (indices only)', 'Max-unpool with saved indices'],
}).set_index('Metric')

print('\n=== FINAL COMPARISON TABLE ===')
print(final_summary.to_string())
final_summary

---
## 13. Interpretation & Recommendations

### Why U-Net typically outperforms SegNet on MA island boundaries

MA islands have **very fine, irregular boundaries** — exactly the scenario
where U-Net's full skip connections give it an advantage over SegNet's
index-only upsampling:

- **U-Net skip connections** transfer complete high-resolution feature maps
  from encoder to decoder at each depth level. This allows the decoder to
  recover fine spatial detail that was lost during pooling.

- **SegNet max-pool indices** only record *where* the max was in each 2×2
  window, not *what the neighbourhood looked like*. Unpooling produces a
  sparse map that must be filled in purely through learned convolutions,
  without access to the rich feature context that U-Net skips provide.

For coarser segmentation tasks (e.g., road segmentation, large organ
detection), SegNet is often competitive and offers lower memory usage.
For microscopy with small, irregular structures like MA islands, U-Net
is the more suitable architecture.

### Suggested next steps

1. **Attention U-Net** — adds channel/spatial attention gates to the skip
   connections, suppressing irrelevant background features.
2. **TransUNet / Swin-UNet** — replace the CNN encoder with a Vision
   Transformer for global context modelling.
3. **Mask R-CNN** — instance segmentation model that outputs a separate
   mask per detected island, removing the need for POI-guided component
   selection entirely.
4. **Ensemble** — average probability maps from U-Net and SegNet to
   exploit their complementary errors.